# DataMate trên Colab Pro: baseline, so model, thu dữ liệu, fine-tune, nộp bài

**Quy chế mục 4.3 cấm chia sẻ data cuộc thi cho bên thứ ba:** KHÔNG đưa data lên GitHub. Mã ở GitHub (private), data ở Google Drive cá nhân.
Chạy lần lượt từng ô. Mục 5 (smoke test) cho biết giây/câu: đọc nó TRƯỚC khi chạy mục 6 (thời gian = số câu x giây/câu). Ô nào lỗi thì gửi lại thông báo.

Runtime: chọn GPU ở *Runtime > Change runtime type*. Model chính: **Qwen3.5-9B** (9,65 tỷ tham số, bf16 ~19GB) + LoRA.
A100 40GB: nạp bf16 đầy đủ (nhanh, khuyến nghị). L4/T4: tự nén 4-bit khi nạp (vừa VRAM nhưng sinh token chậm hơn nhiều).
Qwen3.5 cần `transformers >= 5.2`, mặc định chạy chế độ suy nghĩ (`<think>`): mã tự tắt cho họ Qwen3 (`enable_thinking=False`).
Engine là `hf` (transformers); vLLM chưa được kiểm với Qwen3.5.

**Engine truy vấn:** repo có hai đường. `--query-engine sql` (DuckDB, mặc định trong mã) được viết và đo với Gemini (prompt đọc nguyên rulebook + README + catalog, đòi đầu ra JSON). Model local ở đây dùng `--query-engine pandas`: notebook tự thêm cờ này qua biến `QE`.


In [ ]:
# 1. Kiểm runtime: GPU, RAM, ổ đĩa
!nvidia-smi
!free -g | head -2
!df -h / | tail -1
import torch
print(torch.__version__, "| CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")


In [ ]:
# 2. Drive + mã nguồn từ GitHub (repo private). Token đọc từ Colab Secrets "GITHUB_TOKEN"; KHÔNG ghi token vào notebook.
import base64, getpass, glob, os, subprocess
from google.colab import drive, userdata
drive.mount("/content/drive")

REPO_URL = "https://github.com/Oszumeiii/ZAIC_2026.git"
BRANCH = "main"
WORKDIR = "/content/ZAIC_2026"

try:
    TOKEN = userdata.get("GITHUB_TOKEN")                 # Secrets (biểu tượng chìa khóa), bật "Notebook access"
except Exception:
    TOKEN = getpass.getpass("Chưa có Secret GITHUB_TOKEN. Dán token (không được lưu): ")

# Gửi token bằng header HTTP để KHÔNG ghi nó vào .git/config (remote URL vẫn sạch)
AUTH = base64.b64encode(f"x-access-token:{TOKEN}".encode()).decode()
GIT = ["git", "-c", f"http.extraHeader=Authorization: Basic {AUTH}"]

def run(args, cwd=None):
    """Chạy lệnh, in kết quả với token đã che; lỗi thì dừng với thông báo rõ."""
    r = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    text = (r.stdout + r.stderr).replace(TOKEN, "***").replace(AUTH, "***").strip()
    if text:
        print(text[-1200:])
    if r.returncode:
        raise SystemExit("LỖI khi chạy git (xem thông báo trên). Thường do: token sai/hết hạn/thiếu quyền repo, "
                         "chưa Authorize SSO của org, hoặc sai tên repo/nhánh.")

if not os.path.isdir(f"{WORKDIR}/.git"):
    run(GIT + ["clone", "--branch", BRANCH, REPO_URL, WORKDIR])
else:                                                    # chạy lại ô này để lấy mã mới nhất
    run(GIT + ["fetch", "origin", BRANCH], cwd=WORKDIR)
    run(["git", "checkout", BRANCH], cwd=WORKDIR)
    run(GIT + ["pull", "--ff-only", "origin", BRANCH], cwd=WORKDIR)

# tìm thư mục gốc chứa app/main.py (phòng khi mã nằm trong thư mục con của repo)
hits = glob.glob(f"{WORKDIR}/**/app/main.py", recursive=True)
assert hits, f"Không thấy app/main.py trong {WORKDIR}: mã datamate_agent đã được đẩy lên nhánh {BRANCH} chưa?"
ROOT = os.path.dirname(os.path.dirname(hits[0]))
os.chdir(ROOT)
print("ROOT:", ROOT)
print("commit:", subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())


In [ ]:
# 3. Cài thư viện. Qwen3.5 cần transformers >= 5.2 (nếu ô này nâng bản mà trước đó đã import transformers, hãy Runtime > Restart session rồi chạy lại từ ô 2).
# Mã trên main import duckdb/jsonschema/xlrd/dotenv/google-genai ngay lúc nạp (kể cả khi chạy engine pandas), nên phải cài đủ.
# requirements.txt của repo chưa dùng được nguyên bản trên Colab (kéo vllm), nên liệt kê tường minh:
!pip -q install -U "transformers>=5.2" pytest openpyxl accelerate bitsandbytes peft huggingface_hub "duckdb>=1.4,<2" "jsonschema>=4" "xlrd>=2.0.1" python-dotenv "google-genai>=1.0.0"
# Tăng tốc lớp linear-attention của Qwen3.5 (không cài được thì transformers dùng bản torch thuần, chậm hơn nhưng vẫn đúng):
!pip -q install flash-linear-attention || echo "KHÔNG cài được flash-linear-attention: lớp linear-attention sẽ chạy bản torch (chậm hơn)"
ENGINE = "hf"      # Qwen3.5 chạy bằng transformers; vLLM chưa kiểm với model này
print("engine:", ENGINE)


In [ ]:
# 4. Cấu hình đường dẫn. Data nằm TRONG Drive cá nhân, không nằm trong repo.
import glob, os
D = "/content/drive/MyDrive"
DATA = f"{D}/ZAIC_2026"                     # thư mục trên Drive chứa data DataMate (SỬA nếu khác)
OUT = f"{D}/datamate_out"                   # cache, log, tóm tắt, submission: ở Drive nên bị ngắt session vẫn còn
MERGED = "/content/merged"                  # chỉ dùng nếu muốn gộp LoRA ra một model riêng (~19GB cho 9B bf16): để ổ cục bộ, Drive thường không đủ chỗ
TIME_BUDGET = 1800                          # giây tối đa cho MỖI lần đo (hết giờ thì dừng và chấm phần đã chạy; đừng để treo hàng giờ)

# Tự tìm mọi thư mục có questions.jsonl dưới DATA: có answers.jsonl là TRAIN, không có là TEST (public test)
found = {"train": [], "test": []}
for q in sorted(glob.glob(f"{DATA}/**/questions.jsonl", recursive=True)):
    d = os.path.dirname(q)
    found["train" if os.path.exists(f"{d}/answers.jsonl") else "test"].append(d)
print("Nội dung", DATA, ":", sorted(os.listdir(DATA)) if os.path.isdir(DATA) else "KHÔNG TỒN TẠI, sửa DATA")
for kind, dirs in found.items():
    for d in dirs:
        n_q = sum(1 for _ in open(f"{d}/questions.jsonl", encoding="utf-8"))
        n_biz = len(os.listdir(f"{d}/data")) if os.path.isdir(f"{d}/data") else 0
        print(f"{kind}: {d} | {n_q} câu | {n_biz} business | rulebook.md: {os.path.exists(f'{d}/rulebook.md')}")
assert found["train"], ("Không thấy questions.jsonl (có answers.jsonl) dưới DATA: dữ liệu DataMate chưa giải nén/để đúng chỗ. "
                        "Cần thư mục có questions.jsonl, answers.jsonl, rulebook.md, calendar.csv và data/<business>/.")
TRAIN = found["train"][0]
TEST = found["test"][0] if found["test"] else f"{DATA}/public_test"
if not found["test"]:
    print("CẢNH BÁO: chưa thấy public test (cần cho mục 12). Các mục 1-11 vẫn chạy được trên train.")
print("TRAIN =", TRAIN, "\nTEST  =", TEST, "\n(kỳ vọng: train 3000 câu / 30 business, test 1000 câu / 14 business)")
QE = "--query-engine pandas"                 # model local chạy engine pandas; engine SQL hiện chỉ viết cho Gemini (xem docs/sql-pipeline.md)
MODEL = "Qwen/Qwen3.5-9B"                    # model gốc; LoRA (nếu có) gắn thêm bằng --local-lora
LOCAL = f"--local-model {MODEL} --local-engine {ENGINE}"
COMMON = f"--train-dir {TRAIN} --test-dir {TEST} --out-dir {OUT} --backend local {QE} {LOCAL}"

# ---- SỐ CÂU CHẠY: SỬA Ở ĐÂY ----
# Train có 30 business (~100 câu/business). HOLDOUT_N business bị GIỮ LẠI để đo; phần còn lại là FIT (thu dữ liệu, few-shot, SFT).
# PER_BIZ = lấy N câu ĐẦU của mỗi business; None = lấy HẾT câu của business đó. Thử nhanh để nhỏ, đo chính thức thì None.
HOLDOUT_N = 6
HOLDOUT_PER_BIZ = 5           # 6 business x 5 = 30 câu: CHỈ để kiểm tốc độ/luồng chạy (sai số PA ~ +-0,18). Đọc 's/câu' ở mục 5 rồi mới tăng:
                              # 15 -> 90 câu (+-0,10), None -> ~600 câu (+-0,04). Thời gian ước tính = số câu x s/câu
FIT_PER_BIZ = 40              # 24 business x 40 = ~960 câu để thu dữ liệu. None -> ~2400 câu (cần nhiều giờ)
HOLD = f"--subset holdout --holdout-n {HOLDOUT_N}" + (f" --per-biz {HOLDOUT_PER_BIZ}" if HOLDOUT_PER_BIZ else "")
FIT = f"--subset fit --holdout-n {HOLDOUT_N}" + (f" --per-biz {FIT_PER_BIZ}" if FIT_PER_BIZ else "")
print("holdout:", HOLD, "| fit:", FIT)


In [ ]:
# 4a. Kiểm bản mã: cờ --query-engine phải thật sự có hiệu lực (từng bị mất khi gộp nhánh).
# Nếu cờ bị bỏ qua, mọi lần chạy rơi vào engine SQL viết cho Gemini; model local sẽ gần như abstain hết.
import argparse
from app.main import add_common_args, config_from_args
_args = add_common_args(argparse.ArgumentParser()).parse_args(["--backend", "local", "--query-engine", "pandas"])
assert config_from_args(_args).query_engine == "pandas", ("app/main.py đang làm rơi cờ --query-engine: sửa config_from_args "
                                                          "(thêm query_engine, database_dir, ... vào mapping) rồi chạy lại từ ô 2")
print("query_engine = pandas: OK")


In [ ]:
# 4b. Chạy test (không cần GPU): không được còn failed/error (số test đổi theo nhánh main)
!python -m pytest -q


## 5. Smoke test: 3 câu, đo giây/câu và VRAM (chạy `!nvidia-smi` ở ô khác trong lúc chạy)

In [ ]:
!python -m evaluation.run_eval {COMMON} --max-biz 1 --per-biz 3 --run-name smoke

## 6. Baseline trên HOLDOUT (6 business giữ lại): so các model bằng PA, không bằng danh tiếng
Mỗi model chạy bằng một tiến trình riêng để giải phóng VRAM. Cache riêng theo `--run-name`.
Chênh lệch nhỏ hơn sai số (in ở ô 7) thì chưa kết luận được: tăng số câu holdout hoặc dùng thêm business.

In [ ]:
MODELS = [
    ("qwen35_9b", LOCAL),
    # Muốn so thêm: ("qwen25coder14b", f"--local-model Qwen/Qwen2.5-Coder-14B-Instruct --local-engine {ENGINE}"),
]
for name, flags in MODELS:
    !python -m evaluation.run_eval --train-dir {TRAIN} --test-dir {TEST} --out-dir {OUT} --backend local {QE} {flags} {HOLD} --time-budget {TIME_BUDGET} --run-name {name}


In [ ]:
# 6a. Vì sao nhiều câu abstain? In lý do lỗi + một đoạn code model sinh, từ cache của một lần chạy
import json, glob
RUN = "qwen35_9b"                         # SỬA: tên run cần soi (cùng --run-name ở trên)
for f in glob.glob(f"{OUT}/logs/llm_cache_*{RUN}*.jsonl"):
    rows = [json.loads(l) for l in open(f, encoding="utf-8")]
    print(f, "|", len(rows), "câu")
    for r in rows[:3]:
        print("- id", r["id"], "| agree", r["agree"], "| decision", r["decision"], "| errors", r.get("errors"))
        for s in (r.get("samples") or [])[:2]:
            print("    mẫu:", None if s is None else (s["type"], s["canon"], (s["code"] or "")[:200].replace("\n", " ⏎ ")))


In [ ]:
# 7. So sánh (kèm cảnh báo khi chênh lệch nằm trong sai số)
!python -m evaluation.compare --out-dir {OUT}


## 8. Các cờ nên A/B trên holdout (đổi MỘT thứ mỗi lần, cùng model)
`--no-helpers`, `--no-hints`, `--rulebook-select`, `--n-samples 5 --agree-min 3`, `--exemplars <kho>`. Giữ cái nào làm PA tăng vượt sai số.

In [ ]:
BEST = LOCAL                                      # model dùng cho A/B (đổi nếu model khác thắng ở ô 7)
for name, flags in [("best_nohelpers", "--no-helpers"), ("best_rulebooksel", "--rulebook-select"), ("best_n5", "--n-samples 5 --agree-min 3")]:
    !python -m evaluation.run_eval --train-dir {TRAIN} --test-dir {TEST} --out-dir {OUT} --backend local {QE} {BEST} {flags} {HOLD} --run-name {name}
!python -m evaluation.compare --out-dir {OUT}


## 9. Thu dữ liệu SFT từ chính model (nhóm FIT, KHÔNG chạm holdout)
Sinh đủ N mẫu cho mỗi câu, giữ mọi mẫu cho ra đúng đáp án chuẩn (sạch, đúng ngay lần đầu). Chỉ dùng tập train.
Có thể chạy làm nhiều lần (tăng `--max-biz`/`--per-biz`) vì cache nối tiếp được.

In [ ]:
COLLECT_BUDGET = 5400    # giây mỗi lần chạy; cache nối tiếp được, chạy lại ô này để thu thêm
!python -m evaluation.run_eval {COMMON} {FIT} --time-budget {COLLECT_BUDGET} --n-samples 4 --agree-min 2 --collect-all --export-sft --sft-all --build-exemplars {OUT}/bank.jsonl --run-name collect1
!wc -l {OUT}/sft_train.jsonl


## 10. Fine-tune LoRA trên Qwen3.5-9B (chạy `--dry-run` trước để xem bao nhiêu mẫu vừa độ dài)
Base phải là model GỐC (không phải bản AWQ). `--quant auto`: A100 huấn luyện LoRA trên bf16 đầy đủ, L4/T4 tự chuyển QLoRA 4-bit. Mẫu quá dài bị bỏ (không cắt); nếu dry-run báo gần như mọi mẫu bị loại thì hạ cỡ prompt hoặc tăng `--max-len`.
Chưa chạy thử trên GPU: bộ nhớ khi huấn luyện 8192 token trên 9B chưa đo, nếu hết VRAM hãy giảm `--max-len` (vd 6144).


In [ ]:
BASE = "Qwen/Qwen3.5-9B"
!python -m training.finetune_qlora --data {OUT}/sft_train.jsonl --base {BASE} --out {OUT}/lora --max-len 8192 --dry-run


In [ ]:
!python -m training.finetune_qlora --data {OUT}/sft_train.jsonl --base {BASE} --out {OUT}/lora --max-len 8192
# Adapter (~vài trăm MB) nằm ở Drive. Đo và nộp bằng --local-lora {OUT}/lora: engine hf tự gộp LoRA vào trọng số khi nạp.


In [ ]:
# 11. Đo model đã fine-tune trên CÙNG holdout rồi so với bản gốc (gắn LoRA trực tiếp, không cần gộp)
!python -m evaluation.run_eval {COMMON} --local-lora {OUT}/lora {HOLD} --time-budget {TIME_BUDGET} --run-name finetuned
!python -m evaluation.compare --out-dir {OUT}


## 12. Dự đoán public test (1000 câu) -> submission.jsonl đủ id, có kiểm định. Nộp tối đa 5 lượt/ngày.

In [ ]:
FINAL = LOCAL                                 # nếu bản fine-tune thắng: FINAL = LOCAL + f" --local-lora {OUT}/lora"
!python -m app.main --test-dir {TEST} --out-dir {OUT} --backend local {QE} {FINAL} --run-name final
